In [ ]:
# PyTorch 2.x compatibility fix for RecBole checkpoints
# Must be run BEFORE importing RecBole
# Uses a flag to prevent double-patching on re-runs
import torch

if not getattr(torch, "_recbole_patched", False):
    _original_load = torch.load

    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _original_load(*args, **kwargs)

    torch.load = _patched_load
    torch._recbole_patched = True
    print("Applied weights_only=False patch for RecBole compatibility")
else:
    print("PyTorch patch already applied (skipping)")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
import json
import warnings
from pathlib import Path

import pandas as pd

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm

    print(
        "WARNING: tqdm.notebook not available; install ipywidgets for proper notebook progress bars."
    )

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed

warnings.filterwarnings("ignore")

# Base paths
DATA_PATH = Path("../data")
RECBOLE_DATA_PATH = DATA_PATH / "recbole"
HPO_RESULTS_PATH = RECBOLE_DATA_PATH / "hpo_results_seq"
RESULTS_PATH = RECBOLE_DATA_PATH / "evaluation_results"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
FINAL_MODELS_PATH = RECBOLE_DATA_PATH / "final_models_seq"
FINAL_MODELS_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION SWITCHES
# =============================================================================
RETRAIN_FINAL_MODELS = False  # Force retrain even if valid checkpoints exist
ALLOW_STALE_CHECKPOINTS = False  # Load checkpoint even if metadata mismatches
RANKED_LIST_TOP_N = 100  # Save at least top-N ranked items per user
METRIC_CUTOFFS = [1, 5, 10]  # Accuracy and beyond-accuracy cutoffs

print("Configuration:")
print(f"  RETRAIN_FINAL_MODELS={RETRAIN_FINAL_MODELS}")
print(f"  ALLOW_STALE_CHECKPOINTS={ALLOW_STALE_CHECKPOINTS}")
print(f"  RANKED_LIST_TOP_N={RANKED_LIST_TOP_N}")
print(f"  METRIC_CUTOFFS={METRIC_CUTOFFS}")
print(f"  RESULTS_PATH={RESULTS_PATH}")

In [ ]:
def get_device() -> str:
    """Get best available device: CUDA > CPU (RecBole does not support MPS)."""
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = get_device()
print(f"Using device: {DEVICE}")

In [ ]:
with open(HPO_RESULTS_PATH / "best_hyperparameters.json") as f:
    best_hyperparams = json.load(f)

print("Loaded best hyperparameters for:")
for model_name in best_hyperparams:
    print(f"  - {model_name}")

In [ ]:
with open(RECBOLE_DATA_PATH / "redial_seq" / "evaluation_targets.json") as f:
    targets_data = json.load(f)

test_targets = {int(k): v for k, v in targets_data["test_targets"].items()}
test_user_ids = [int(uid) for uid in targets_data["test_user_ids"]]

with open(RECBOLE_DATA_PATH / "redial_seq" / "id_to_title.json") as f:
    id_to_title = {int(k): v for k, v in json.load(f).items()}

dialogue_to_user = {
    str(k): int(v) for k, v in targets_data.get("test_dialogue_to_user", {}).items()
}

print(f"Loaded {len(test_targets)} test users with targets")
print(f"Loaded {len(dialogue_to_user)} dialogue_id mappings")
print(f"Loaded {len(id_to_title)} item ID to title mappings")

In [ ]:
import csv
from math import ceil


def _count_users_and_interactions(p):
    users = set()
    n = 0
    with open(p, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        user_key = [k for k in reader.fieldnames if k.startswith("user_id")][0]
        for row in reader:
            users.add(row[user_key])
            n += 1
    return len(users), n


train_path = RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.train.inter"
valid_path = RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.valid.inter"
test_path = RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.test.inter"

train_users, train_n = _count_users_and_interactions(train_path)
valid_users, valid_n = _count_users_and_interactions(valid_path)
test_users, test_n = _count_users_and_interactions(test_path)

print("Split counts:")
print(f"  Train: {train_n} interactions, {train_users} users")
print(f"  Valid: {valid_n} interactions, {valid_users} users")
print(f"  Test:  {test_n} interactions, {test_users} users")

# Verify warm-start
train_user_set = set()
with open(train_path, newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    user_key = [k for k in reader.fieldnames if k.startswith("user_id")][0]
    for row in reader:
        train_user_set.add(row[user_key])

test_user_set = set()
with open(test_path, newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    user_key = [k for k in reader.fieldnames if k.startswith("user_id")][0]
    for row in reader:
        test_user_set.add(row[user_key])

test_in_train = test_user_set & train_user_set
print(f"\nWarm-start: {len(test_in_train)}/{len(test_user_set)} test users in train")

train_batches = ceil(train_n / 2048)
print(f"Expected training batches per epoch: ~{train_batches}")

In [ ]:
# BERT4Rec dropped: uses masked item prediction which is incompatible
# with our pre-augmented sliding-window data format (causes NaN loss)
SEQUENTIAL_MODELS = ["SASRec", "GRU4Rec", "NARM", "SRGNN"]
ALL_MODELS = SEQUENTIAL_MODELS

print(f"Models to evaluate: {len(ALL_MODELS)}")
print(f"Sequential models: {SEQUENTIAL_MODELS}")

In [ ]:
def get_model_config(model_name: str, best_params: dict) -> dict:
    """Build sequential model configuration with best hyperparameters."""
    return {
        # Dataset settings (sequential)
        "data_path": str(RECBOLE_DATA_PATH),
        "dataset": "redial_seq",
        "benchmark_filename": ["train", "valid", "test"],
        # Field definitions
        "USER_ID_FIELD": "user_id",
        "ITEM_ID_FIELD": "item_id",
        "LIST_SUFFIX": "_list",
        "ITEM_LIST_LENGTH_FIELD": "item_length",
        "load_col": {
            "inter": ["user_id", "item_id", "item_id_list"],
        },
        # Sequential-specific settings
        "MAX_ITEM_LIST_LENGTH": 50,
        "alias_of_item_id": ["item_id_list"],
        "repeatable": True,
        # Training settings
        "epochs": 100,
        "train_batch_size": 2048,
        "eval_batch_size": 2048,
        "learning_rate": 0.001,
        "stopping_step": 5,
        # Evaluation settings
        "eval_args": {
            "group_by": "user",
            "order": "TO",
            "split": {"LS": "valid_and_test"},
            "mode": "full",
        },
        "metrics": ["Recall", "MRR", "NDCG", "Hit", "Precision"],
        "topk": METRIC_CUTOFFS,
        "valid_metric": "NDCG@10",
        # No negative sampling (CE loss models)
        "train_neg_sample_args": None,
        # Device and reproducibility
        "device": DEVICE,
        "seed": SEED,
        "reproducibility": True,
        "show_progress": False,
        # Logging
        "log_wandb": False,
        "state": "INFO",
        "model": model_name,
        # Best hyperparameters from HPO
        **best_params,
    }

In [ ]:
import shutil

for cache_dir in ["dataset", "log", "saved"]:
    p = Path(cache_dir)
    if p.exists():
        shutil.rmtree(p)
        print(f"Cleared {cache_dir}/")
    else:
        print(f"{cache_dir}/ not found (OK)")

Path("saved").mkdir(exist_ok=True)
print("RecBole cache cleared.")

In [ ]:
from stability.checkpoint import (
    build_checkpoint_metadata,
    save_checkpoint_metadata,
    validate_checkpoint_metadata,
)


def _build_expected_metadata(model_name: str, config_dict: dict) -> dict:
    """Build expected metadata dict for validation."""
    return build_checkpoint_metadata(
        model_name=model_name,
        model_family="sequential",
        dataset_name="redial_seq",
        config=config_dict,
        best_hyperparameters=best_hyperparams.get(model_name, {}).get("params", {}),
        seed=SEED,
        source_hyperparameter_file=str(HPO_RESULTS_PATH / "best_hyperparameters.json"),
        source_data_files=[
            str(RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.train.inter"),
            str(RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.valid.inter"),
            str(RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.test.inter"),
        ],
    )


def _load_checkpoint(model_name: str, config_dict: dict) -> dict:
    """Load model, dataset, and trainer from an existing checkpoint."""
    checkpoint_path = FINAL_MODELS_PATH / f"{model_name}.pth"
    config = Config(model=model_name, config_dict=config_dict)
    init_seed(config["seed"], config["reproducibility"])
    dataset = create_dataset(config)
    train_data, _valid_data, test_data = data_preparation(config, dataset)
    model = get_model(config["model"])(config, train_data._dataset).to(config["device"])
    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
    if model_name in ("Pop", "Random"):
        # These baselines have no useful learned state_dict; fitting restores their
        # train-data-derived statistics before scoring.
        trainer.fit(train_data, show_progress=False)
    else:
        state_dict = torch.load(checkpoint_path, map_location=config["device"])
        model.load_state_dict(state_dict)
    model.eval()
    return {
        "status": "success",
        "decision": "loaded",
        "model": model,
        "trainer": trainer,
        "dataset": train_data._dataset,
        "test_data": test_data,
        "best_valid_score": None,
        "test_result": None,
    }

In [ ]:
def train_and_evaluate(model_name: str, config_dict: dict) -> dict:
    """Train model on train split and evaluate on test set via RecBole."""
    try:
        config = Config(model=model_name, config_dict=config_dict)
        init_seed(config["seed"], config["reproducibility"])

        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = get_model(config["model"])(config, train_data._dataset).to(
            config["device"]
        )
        trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

        best_valid_score, best_valid_result = trainer.fit(
            train_data, valid_data, verbose=False, show_progress=False
        )

        test_result = trainer.evaluate(test_data)

        return {
            "status": "success",
            "decision": "trained",
            "best_valid_score": float(best_valid_score),
            "test_result": {k: float(v) for k, v in test_result.items()},
            "model": model,
            "trainer": trainer,
            "dataset": train_data._dataset,
            "test_data": test_data,
        }
    except Exception as e:
        print(f"  Error: {e}")
        return {"status": "failed", "error": str(e)}

In [ ]:
def train_or_load_model(model_name: str, config_dict: dict) -> dict:
    """Train model or load from checkpoint based on metadata validation."""
    checkpoint_path = FINAL_MODELS_PATH / f"{model_name}.pth"
    expected_metadata = _build_expected_metadata(model_name, config_dict)
    decision = "train"

    if not RETRAIN_FINAL_MODELS and checkpoint_path.exists():
        val_result = validate_checkpoint_metadata(expected_metadata, checkpoint_path)
        if val_result["status"] == "match":
            decision = "load"
        elif val_result["status"] == "missing":
            print(f"  {model_name}: metadata missing -> will retrain")
            decision = "train"
        elif val_result["status"] == "mismatch":
            if ALLOW_STALE_CHECKPOINTS:
                print(f"  {model_name}: metadata mismatch -> loading stale checkpoint")
                decision = "load"
            else:
                print(f"  {model_name}: metadata mismatch -> will retrain")
                for reason in val_result["reasons"]:
                    print(f"    - {reason}")
                decision = "train"

    if decision == "load":
        try:
            result = _load_checkpoint(model_name, config_dict)
            print(f"  Loaded {model_name} from checkpoint")
            return result
        except Exception as e:
            print(f"  Failed to load {model_name}: {e}")
            decision = "train"

    result = train_and_evaluate(model_name, config_dict)
    if result["status"] == "success":
        torch.save(result["model"].state_dict(), checkpoint_path)
        save_checkpoint_metadata(expected_metadata, checkpoint_path)
        print(f"  Saved checkpoint and metadata for {model_name}")
    return result

In [ ]:
recbole_results = {}

for model_name in tqdm(ALL_MODELS, desc="Models"):
    print(f"\n{'=' * 60}")
    print(f"{model_name}")
    print(f"{'=' * 60}")

    hp = best_hyperparams.get(model_name, {})
    best_params = hp.get("params", {})
    config_dict = get_model_config(model_name, best_params)
    config_dict["model"] = model_name

    result = train_or_load_model(model_name, config_dict)
    recbole_results[model_name] = result

    if result["status"] == "success" and result.get("test_result"):
        print(f"\n{model_name} Test Results:")
        for metric, value in result["test_result"].items():
            if "@10" in metric:
                print(f"  {metric}: {value:.4f}")
    elif result["status"] == "success":
        print(f"\n{model_name}: loaded from checkpoint (no RecBole test result)")

saved = sum(1 for r in recbole_results.values() if r["status"] == "success")
print(f"\n{'=' * 60}")
print(f"Successfully prepared {saved}/{len(ALL_MODELS)} models")

In [ ]:
from stability.evaluation import (
    aggregate_accuracy_metrics,
    average_popularity_at_k,
    compute_per_user_accuracy,
    compute_tail_items,
    genre_coverage_at_k,
    genre_entropy_at_k,
    gini_index_at_k,
    item_coverage_at_k,
    load_catalog,
    load_genre_mapping,
    load_popularity,
    ranking_rows_to_dataframe,
    shannon_entropy_at_k,
    tail_percentage_at_k,
)

item_path = RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.item"
train_inter_path = RECBOLE_DATA_PATH / "redial_seq" / "redial_seq.train.inter"
metadata_path = DATA_PATH / "processed" / "movies_metadata_tmdb.csv"

id_to_title_catalog, catalog_item_ids = load_catalog(item_path)
popularity_counts = load_popularity(train_inter_path, catalog_item_ids)
tail_item_ids = compute_tail_items(popularity_counts, tail_fraction=0.1)
genre_mapping = load_genre_mapping(metadata_path, id_to_title_catalog)

print(f"Catalog size: {len(catalog_item_ids)}")
print(f"Tail items: {len(tail_item_ids)}")
print(f"Genres mapped: {sum(1 for g in genre_mapping.values() if g)}")

In [ ]:
import re

title_to_id = {v: k for k, v in id_to_title.items()}


def match_title_fuzzy(title: str, title_to_id: dict) -> int | None:
    """Match title with fuzzy fallback."""
    if title in title_to_id:
        return title_to_id[title]
    title_lower = title.strip().lower()
    for key, val in title_to_id.items():
        if key.lower() == title_lower:
            return val
    return None


# Load test templates for dialogue_id matching
test_templates = []
with open(DATA_PATH / "processed" / "test_prompt_templates.jsonl") as f:
    for line in f:
        test_templates.append(json.loads(line))

CANDIDATE_SIZES = [250, 500, 1000]
candidate_sets = {}
unmatched_records = 0

for n_cand in CANDIDATE_SIZES:
    filepath = DATA_PATH / "processed" / f"test_prompt_examples_c{n_cand}_r10.jsonl"
    if not filepath.exists():
        print(f"WARNING: Candidate file not found: {filepath}")
        continue

    candidates = {}
    unmatched_titles = set()

    template_indices = [
        ti
        for ti, tmpl in enumerate(test_templates)
        if tmpl.get("recommended_accepted", [])
    ]

    with open(filepath) as f:
        for idx, line in enumerate(f):
            record = json.loads(line)
            if idx >= len(template_indices):
                break
            template = test_templates[template_indices[idx]]
            did = str(template.get("dialogue_id", ""))
            user_id = dialogue_to_user.get(did)
            if user_id is None:
                unmatched_records += 1
                continue

            user_content = record["messages"][1]["content"]
            matches = list(re.finditer(r"<CANDIDATES>", user_content))

            if len(matches) >= 2:
                start = matches[1].start() + len("<CANDIDATES>")
                end_match = re.search(r"</CANDIDATES>|<[A-Z]", user_content[start:])
                end = start + end_match.start() if end_match else len(user_content)
                candidates_block = user_content[start:end]
            elif len(matches) == 1:
                start = matches[0].start() + len("<CANDIDATES>")
                end_match = re.search(r"</CANDIDATES>|<[A-Z]", user_content[start:])
                end = start + end_match.start() if end_match else len(user_content)
                candidates_block = user_content[start:end]
            else:
                candidates_block = ""

            candidate_titles = [
                line.strip() for line in candidates_block.split("\n") if line.strip()
            ]
            candidate_ids = []
            for title in candidate_titles:
                item_id = match_title_fuzzy(title, title_to_id)
                if item_id is not None:
                    candidate_ids.append(item_id)
                else:
                    unmatched_titles.add(title)

            if candidate_ids:
                candidates[user_id] = candidate_ids

    if candidates:
        candidate_sets[n_cand] = candidates
        print(f"Loaded {n_cand} candidates: {len(candidates)} users")
        if unmatched_titles:
            print(f"  Unmatched titles: {len(unmatched_titles)}")

if unmatched_records:
    print(f"\nSkipped {unmatched_records} records with no matching RecBole user")
print(f"Candidate sets loaded: {list(candidate_sets.keys())}")

In [ ]:
from recbole.data.interaction import Interaction


def get_user_train_items(dataset) -> dict:
    """Get original item IDs each user interacted with in training."""
    uid_field = dataset.uid_field
    iid_field = dataset.iid_field
    uid_id2token = dataset.field2id_token[uid_field]
    iid_id2token = dataset.field2id_token[iid_field]

    user_train_items = {}
    for i in range(len(dataset.inter_feat[uid_field])):
        uid_token = uid_id2token[dataset.inter_feat[uid_field][i].item()]
        iid_token = iid_id2token[dataset.inter_feat[iid_field][i].item()]
        if uid_token == "" or iid_token == "":
            continue
        try:
            orig_uid = int(uid_token)
            orig_iid = int(iid_token)
        except ValueError:
            continue
        user_train_items.setdefault(orig_uid, set()).add(orig_iid)
    return user_train_items


def _unwrap_interaction_dataset(eval_data):
    """Return the RecBole dataset object behind an eval dataloader, if needed."""
    if eval_data is None:
        return None
    if hasattr(eval_data, "inter_feat"):
        return eval_data
    if hasattr(eval_data, "dataset") and hasattr(eval_data.dataset, "inter_feat"):
        return eval_data.dataset
    if hasattr(eval_data, "_dataset") and hasattr(eval_data._dataset, "inter_feat"):
        return eval_data._dataset
    return eval_data


def get_model_item_scores(model, dataset, test_users: list, test_dataset=None) -> dict:
    """Get item scores from trained sequential model for all test users.

    For sequential models, extracts the frozen item sequence from test_dataset
    and passes it into full_sort_predict along with the user_id.

    Returns: {original_user_id: {original_item_id: score}}
    """
    model.eval()
    scores = {}
    test_dataset = _unwrap_interaction_dataset(test_dataset)

    user_token2id = dataset.field2token_id[dataset.uid_field]
    item_id2token = dataset.field2id_token[dataset.iid_field]
    n_items = dataset.item_num

    # Build user->sequence mapping from test dataset for sequential models
    user_to_seq = {}
    seq_field = getattr(model, "ITEM_SEQ", None)
    len_field = getattr(model, "ITEM_SEQ_LEN", None)

    if test_dataset is not None and seq_field is not None and len_field is not None:
        uid_field = dataset.uid_field
        inter_feat = test_dataset.inter_feat
        for i in range(len(inter_feat[uid_field])):
            uid = inter_feat[uid_field][i].item()
            if uid not in user_to_seq:
                user_to_seq[uid] = {
                    "seq": inter_feat[seq_field][i],
                    "len": inter_feat[len_field][i],
                }
        print(f"    Built sequence mapping for {len(user_to_seq)} test users")
    else:
        print("    No test_dataset provided or model has no ITEM_SEQ field")

    missing_users = 0
    error_count = 0
    score_offset = None

    with torch.no_grad():
        for original_user_id in tqdm(
            test_users, desc="Getting user scores", leave=False
        ):
            internal_user_id = user_token2id.get(str(original_user_id))
            if internal_user_id is None:
                missing_users += 1
                continue

            try:
                user_tensor = torch.LongTensor([internal_user_id]).to(model.device)

                if hasattr(model, "full_sort_predict"):
                    interaction_dict = {dataset.uid_field: user_tensor}

                    # For sequential models: inject the item sequence
                    if seq_field is not None and internal_user_id in user_to_seq:
                        seq_data = user_to_seq[internal_user_id]
                        interaction_dict[seq_field] = (
                            seq_data["seq"].unsqueeze(0).to(model.device)
                        )
                        interaction_dict[len_field] = torch.LongTensor(
                            [
                                seq_data["len"].item()
                                if hasattr(seq_data["len"], "item")
                                else int(seq_data["len"])
                            ]
                        ).to(model.device)

                    interaction = Interaction(interaction_dict)
                    interaction = interaction.to(model.device)
                    item_scores = model.full_sort_predict(interaction)
                    item_scores = item_scores.cpu().numpy().flatten()
                else:
                    item_tensor = torch.arange(n_items).to(model.device)
                    user_tensor_expanded = user_tensor.expand(n_items)
                    interaction = Interaction(
                        {
                            dataset.uid_field: user_tensor_expanded,
                            dataset.iid_field: item_tensor,
                        }
                    )
                    interaction = interaction.to(model.device)
                    item_scores = model.predict(interaction).cpu().numpy()

                if score_offset is None:
                    token_len = len(item_id2token)
                    score_len = len(item_scores)
                    if score_len == token_len:
                        score_offset = 0
                    elif score_len == token_len - 1:
                        score_offset = 1
                    else:
                        score_offset = 0
                        print(
                            f"  WARNING: score length {score_len} != token length {token_len}"
                        )
                    print(
                        f"  Score vector: len={score_len}, tokens={token_len}, offset={score_offset}"
                    )

                user_score_dict = {}
                for internal_item_id, score in enumerate(item_scores):
                    token_index = internal_item_id + score_offset
                    if token_index >= len(item_id2token):
                        break
                    original_item_token = item_id2token[token_index]
                    if original_item_token != "":
                        try:
                            original_item_id = int(original_item_token)
                            user_score_dict[original_item_id] = float(score)
                        except ValueError:
                            continue

                scores[original_user_id] = user_score_dict

            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    print(
                        f"  Error for user {original_user_id}: {type(e).__name__}({e!r})"
                    )
                continue

    if missing_users:
        print(f"  WARNING: {missing_users} users not found in dataset")
    if error_count:
        print(f"  WARNING: {error_count} users failed during scoring")
    print(f"  Successfully scored {len(scores)} users")
    return scores

In [ ]:
all_ranked_rows = []  # Per-user ranked-list artifacts
all_aggregate_rows = []  # Long-form aggregate metrics
run_manifest = {
    "timestamp": None,
    "source_notebook": "11_recbole_evaluation_sequential.ipynb",
    "models": [],
    "config": {
        "retrain_final_models": RETRAIN_FINAL_MODELS,
        "allow_stale_checkpoints": ALLOW_STALE_CHECKPOINTS,
        "ranked_list_top_n": RANKED_LIST_TOP_N,
        "metric_cutoffs": METRIC_CUTOFFS,
    },
    "dataset_paths": {
        "train": str(train_path),
        "valid": str(valid_path),
        "test": str(test_path),
        "catalog": str(item_path),
        "metadata": str(metadata_path),
    },
    "hyperparameter_source": str(HPO_RESULTS_PATH / "best_hyperparameters.json"),
    "checkpoint_paths": {},
    "checkpoint_decisions": {},
    "output_files": [],
    "git_commit": None,
}

try:
    from stability.checkpoint import _get_git_commit

    run_manifest["git_commit"] = _get_git_commit()
except Exception:  # noqa: S110
    pass

for model_name in tqdm(ALL_MODELS, desc="Evaluating models"):
    result = recbole_results.get(model_name, {})
    if result.get("status") != "success":
        continue

    print(f"\n{'=' * 60}")
    print(f"Evaluating: {model_name}")
    print(f"{'=' * 60}")

    model = result["model"]
    dataset = result["dataset"]
    test_dataset = result.get("test_data")
    model_scores = get_model_item_scores(
        model, dataset, test_user_ids, test_dataset=test_dataset
    )
    user_train_items = get_user_train_items(dataset)

    # ------------------------------------------------------------------
    # Standalone full-catalog evaluation
    # ------------------------------------------------------------------
    standalone_rows = []
    for user_id in test_user_ids:
        if user_id not in model_scores or user_id not in test_targets:
            continue
        user_scores = model_scores[user_id]
        targets = test_targets[user_id]
        train_items = user_train_items.get(user_id, set())
        sorted_items = sorted(
            (
                (iid, score)
                for iid, score in user_scores.items()
                if iid not in train_items
            ),
            key=lambda x: x[1],
            reverse=True,
        )[:RANKED_LIST_TOP_N]

        ranked_item_ids = [int(iid) for iid, _ in sorted_items]
        ranked_scores = [float(score) for _, score in sorted_items]
        ranked_titles = [id_to_title.get(iid, f"ID:{iid}") for iid in ranked_item_ids]
        ground_truth_item_ids = [int(tid) for tid in targets]
        ground_truth_titles = [id_to_title.get(tid, f"ID:{tid}") for tid in targets]

        standalone_rows.append(
            {
                "model_type": "sequential",
                "model": model_name,
                "eval_method": "standalone",
                "retriever_type": "sequential",
                "retriever_model": model_name,
                "reranker_type": "none",
                "reranker_model": "none",
                "n_candidates": len(catalog_item_ids),
                "user_id": user_id,
                "prompt_idx": -1,
                "ranked_item_ids": ranked_item_ids,
                "ranked_titles": ranked_titles,
                "ranked_scores": ranked_scores,
                "ground_truth_item_ids": ground_truth_item_ids,
                "ground_truth_titles": ground_truth_titles,
            }
        )

    if standalone_rows:
        df_standalone = ranking_rows_to_dataframe(standalone_rows)
        df_standalone = compute_per_user_accuracy(
            df_standalone, k_values=METRIC_CUTOFFS
        )

        # Accuracy aggregate
        acc_agg = aggregate_accuracy_metrics(
            df_standalone,
            k_values=METRIC_CUTOFFS,
            metadata={
                "model_type": "sequential",
                "model": model_name,
                "eval_method": "standalone",
                "retriever_type": "sequential",
                "retriever_model": model_name,
                "reranker_type": "none",
                "reranker_model": "none",
                "n_candidates": len(catalog_item_ids),
            },
        )
        all_aggregate_rows.append(acc_agg)

        # Beyond-accuracy aggregate
        ranked_item_ids_list = df_standalone["ranked_item_ids"].tolist()
        for k in METRIC_CUTOFFS:
            ba_row = {
                "model_type": "sequential",
                "model": model_name,
                "eval_method": "standalone",
                "retriever_type": "sequential",
                "retriever_model": model_name,
                "reranker_type": "none",
                "reranker_model": "none",
                "n_candidates": len(catalog_item_ids),
                "metric": None,
                "k": k,
                "value": None,
            }
            ba_row["metric"] = "item_coverage"
            ba_row["value"] = item_coverage_at_k(
                ranked_item_ids_list, catalog_item_ids, k
            )
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

            ba_row["metric"] = "average_popularity"
            ba_row["value"] = average_popularity_at_k(
                ranked_item_ids_list, popularity_counts, k
            )
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

            ba_row["metric"] = "gini_index"
            ba_row["value"] = gini_index_at_k(ranked_item_ids_list, catalog_item_ids, k)
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

            ba_row["metric"] = "shannon_entropy"
            ba_row["value"] = shannon_entropy_at_k(
                ranked_item_ids_list, catalog_item_ids, k
            )
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

            ba_row["metric"] = "tail_percentage"
            ba_row["value"] = tail_percentage_at_k(
                ranked_item_ids_list, tail_item_ids, k
            )
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

            ba_row["metric"] = "genre_coverage"
            ba_row["value"] = genre_coverage_at_k(
                ranked_item_ids_list, genre_mapping, k
            )
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

            ba_row["metric"] = "genre_entropy"
            ba_row["value"] = genre_entropy_at_k(ranked_item_ids_list, genre_mapping, k)
            all_aggregate_rows.append(pd.DataFrame([ba_row]))

        all_ranked_rows.extend(df_standalone.to_dict("records"))
        m10 = df_standalone["ndcg@10"].mean()
        print(f"  Standalone: NDCG@10={m10:.4f}, users={len(df_standalone)}")

    # ------------------------------------------------------------------
    # CBF+Reranker evaluation
    # ------------------------------------------------------------------
    for n_cand, user_candidates in candidate_sets.items():
        cbf_rows = []
        for user_id, candidate_ids in user_candidates.items():
            if user_id not in model_scores or user_id not in test_targets:
                continue
            user_scores = model_scores[user_id]
            targets = test_targets[user_id]
            candidate_scores = [
                (item_id, user_scores.get(item_id, float("-inf")))
                for item_id in candidate_ids
            ]
            candidate_scores.sort(key=lambda x: x[1], reverse=True)
            topk = candidate_scores[:RANKED_LIST_TOP_N]
            ranked_item_ids = [int(iid) for iid, _ in topk]
            ranked_scores = [float(score) for _, score in topk]
            ranked_titles = [
                id_to_title.get(iid, f"ID:{iid}") for iid in ranked_item_ids
            ]
            ground_truth_item_ids = [int(tid) for tid in targets]
            ground_truth_titles = [id_to_title.get(tid, f"ID:{tid}") for tid in targets]

            cbf_rows.append(
                {
                    "model_type": "sequential",
                    "model": f"CBF+{model_name}",
                    "eval_method": "cbf_reranking",
                    "retriever_type": "cbf",
                    "retriever_model": "CBF",
                    "reranker_type": "sequential",
                    "reranker_model": model_name,
                    "n_candidates": n_cand,
                    "user_id": user_id,
                    "prompt_idx": -1,
                    "ranked_item_ids": ranked_item_ids,
                    "ranked_titles": ranked_titles,
                    "ranked_scores": ranked_scores,
                    "ground_truth_item_ids": ground_truth_item_ids,
                    "ground_truth_titles": ground_truth_titles,
                }
            )

        if cbf_rows:
            df_cbf = ranking_rows_to_dataframe(cbf_rows)
            df_cbf = compute_per_user_accuracy(df_cbf, k_values=METRIC_CUTOFFS)

            acc_agg = aggregate_accuracy_metrics(
                df_cbf,
                k_values=METRIC_CUTOFFS,
                metadata={
                    "model_type": "sequential",
                    "model": f"CBF+{model_name}",
                    "eval_method": "cbf_reranking",
                    "retriever_type": "cbf",
                    "retriever_model": "CBF",
                    "reranker_type": "sequential",
                    "reranker_model": model_name,
                    "n_candidates": n_cand,
                },
            )
            all_aggregate_rows.append(acc_agg)

            ranked_item_ids_list = df_cbf["ranked_item_ids"].tolist()
            for k in METRIC_CUTOFFS:
                ba_row = {
                    "model_type": "sequential",
                    "model": f"CBF+{model_name}",
                    "eval_method": "cbf_reranking",
                    "retriever_type": "cbf",
                    "retriever_model": "CBF",
                    "reranker_type": "sequential",
                    "reranker_model": model_name,
                    "n_candidates": n_cand,
                    "metric": None,
                    "k": k,
                    "value": None,
                }
                ba_row["metric"] = "item_coverage"
                ba_row["value"] = item_coverage_at_k(
                    ranked_item_ids_list, catalog_item_ids, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

                ba_row["metric"] = "average_popularity"
                ba_row["value"] = average_popularity_at_k(
                    ranked_item_ids_list, popularity_counts, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

                ba_row["metric"] = "gini_index"
                ba_row["value"] = gini_index_at_k(
                    ranked_item_ids_list, catalog_item_ids, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

                ba_row["metric"] = "shannon_entropy"
                ba_row["value"] = shannon_entropy_at_k(
                    ranked_item_ids_list, catalog_item_ids, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

                ba_row["metric"] = "tail_percentage"
                ba_row["value"] = tail_percentage_at_k(
                    ranked_item_ids_list, tail_item_ids, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

                ba_row["metric"] = "genre_coverage"
                ba_row["value"] = genre_coverage_at_k(
                    ranked_item_ids_list, genre_mapping, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

                ba_row["metric"] = "genre_entropy"
                ba_row["value"] = genre_entropy_at_k(
                    ranked_item_ids_list, genre_mapping, k
                )
                all_aggregate_rows.append(pd.DataFrame([ba_row]))

            all_ranked_rows.extend(df_cbf.to_dict("records"))
            m10 = df_cbf["ndcg@10"].mean()
            print(
                f"  CBF+{model_name} ({n_cand}): NDCG@10={m10:.4f}, users={len(df_cbf)}"
            )

    run_manifest["models"].append(model_name)
    run_manifest["checkpoint_paths"][model_name] = str(
        FINAL_MODELS_PATH / f"{model_name}.pth"
    )
    run_manifest["checkpoint_decisions"][model_name] = result.get("decision", "unknown")

print(f"\nEvaluation complete. Total ranked rows: {len(all_ranked_rows)}")

In [ ]:
if all_ranked_rows:
    df_all_ranked = ranking_rows_to_dataframe(all_ranked_rows)
    ranked_path = RESULTS_PATH / "seq_ranked_lists.parquet"
    df_all_ranked.to_parquet(ranked_path, index=False)
    run_manifest["output_files"].append(str(ranked_path))
    print(f"Saved {len(df_all_ranked)} ranked-list rows to {ranked_path}")
else:
    print("WARNING: No ranked-list rows to save")

In [ ]:
if all_aggregate_rows:
    df_aggregate = pd.concat(all_aggregate_rows, ignore_index=True)
    # Reorder columns for readability
    first_cols = [
        "model_type",
        "model",
        "eval_method",
        "retriever_type",
        "retriever_model",
        "reranker_type",
        "reranker_model",
        "n_candidates",
        "metric",
        "k",
        "value",
    ]
    other_cols = [c for c in df_aggregate.columns if c not in first_cols]
    df_aggregate = df_aggregate[first_cols + other_cols]

    agg_path = RESULTS_PATH / "seq_aggregate_metrics.parquet"
    df_aggregate.to_parquet(agg_path, index=False)
    run_manifest["output_files"].append(str(agg_path))
    print(f"Saved {len(df_aggregate)} aggregate metric rows to {agg_path}")
else:
    print("WARNING: No aggregate metrics to save")

In [ ]:
from datetime import datetime, timezone

run_manifest["timestamp"] = datetime.now(tz=timezone.utc).isoformat()
manifest_path = RESULTS_PATH / "seq_run_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(run_manifest, f, indent=2, default=str)
run_manifest["output_files"].append(str(manifest_path))
print(f"Saved run manifest to {manifest_path}")

In [ ]:
print("\nArtifact producer finished.")
print("Outputs:")
for p in run_manifest.get("output_files") or []:
    print(f"  - {p}")